In [ ]:
explain_running_pattern(
    history_df,           # 이 유저의 running_record 히스토리 (DataFrame)
    next_month_km=None    # LSTM 팀이 준 예측 값 (없으면 과거만 기반으로 설명)
)


In [1]:
import pandas as pd

def load_user_history_from_db(user_id, engine):
    """
    DB에서 특정 사용자의 러닝 기록을 불러오는 함수.
    XAI와 LSTM 모두 동일하게 사용할 수 있음.
    """
    query = """
        SELECT user_id, start_time, distance_km, pace_km, avg_heart_rate, run_type
        FROM running_record
        WHERE user_id = :uid
        ORDER BY start_time ASC
    """
    df = pd.read_sql(query, engine, params={"uid": user_id})
    return df


In [ ]:
# def explain_running_pattern_from_db(user_id, engine, next_month_km=None):
#     # DB에서 user_id의 러닝 기록들을 자동으로 가져옴
#     # XAI 분석 로직 수행
#     # 설명 문자열 반환


In [2]:
def explain_running_pattern(history_df, next_month_km=None):
    """
    history_df: DB에서 가져온 특정 유저의 러닝 기록 DataFrame
    next_month_km: LSTM 팀이 제공한 다음 달 예측 거리 (optional)
    """

    if len(history_df) < 10:
        return "데이터가 충분하지 않아 패턴 분석이 어렵습니다."

    # 최근 5회 / 그 이전 5회 분리
    recent = history_df.iloc[-5:]
    previous = history_df.iloc[-10:-5]

    explanation = []

    # 1) 거리 변화
    r_dist = recent['distance_km'].mean()
    p_dist = previous['distance_km'].mean()

    if r_dist > p_dist * 1.1:
        explanation.append("📈 최근 평균 러닝 거리가 증가하고 있습니다.")
    else:
        explanation.append("📉 최근 러닝 거리가 다소 감소하고 있습니다.")

    # 2) 페이스 변화
    r_pace = recent['pace_km'].mean()
    p_pace = previous['pace_km'].mean()

    if r_pace < p_pace:
        explanation.append("🏃‍♂️ 최근 페이스가 더 빨라졌어요!")
    else:
        explanation.append("🐢 최근 페이스가 조금 느려졌습니다.")

    # 3) 심박 변화
    r_hr = recent['avg_heart_rate'].mean()
    p_hr = previous['avg_heart_rate'].mean()

    if r_hr < p_hr:
        explanation.append("💓 심박이 더 안정적입니다. 러닝 효율이 좋아졌어요.")
    else:
        explanation.append("💓 최근 러닝 시 심박수가 조금 높게 나타나고 있습니다.")

    # 4) LSTM 예측 연동
    if next_month_km is not None:
        explanation.append(
            f"📅 LSTM 예측에 따르면 다음 달 예상 총 러닝 거리는 약 {next_month_km:.1f} km 입니다."
        )

    return "\n".join(explanation)


In [3]:
def run_xai_for_user(user_id, engine, next_month_km=None):
    # 1) DB에서 사용자 기록 불러오기
    history_df = load_user_history_from_db(user_id, engine)

    # 2) XAI 분석 실행
    result = explain_running_pattern(history_df, next_month_km)

    return result


In [4]:
# 예: LSTM 팀이 만든 예측 함수
next_km = predict_next_distance_for_user(
    user_id=1,
    df=df,
    scaler=scaler,
    model=model
)

# XAI 실행
result = run_xai_for_user(
    user_id=1,
    engine=engine,
    next_month_km=next_km
)

print(result)


NameError: name 'predict_next_distance_for_user' is not defined

In [ ]:
import pandas as pd
import numpy as np

# -------------------------------
# 1) 유틸: 최근/이전 기간 나누기
# -------------------------------
def split_recent_past(df, recent_n=5):
    df = df.sort_values("start_time")
    if len(df) < recent_n * 2:
        # 데이터가 너무 적으면 그냥 전체를 recent로 쓰고, past는 빈 것
        return df, pd.DataFrame([])
    recent = df.iloc[-recent_n:]
    past   = df.iloc[-2*recent_n:-recent_n]
    return recent, past

# -------------------------------
# 2) 기본 통계 계산
# -------------------------------
def compute_basic_stats(df):
    if df.empty:
        return {
            "mean_distance": np.nan,
            "mean_pace": np.nan,
            "mean_hr": np.nan,
            "run_type_ratio": {}
        }
    run_type_ratio = (df["run_type"].value_counts(normalize=True)
                      .to_dict() if "run_type" in df.columns else {})
    return {
        "mean_distance": df["distance_km"].mean(),
        "mean_pace": df["pace_km"].mean(),
        "mean_hr": df["avg_heart_rate"].mean(),
        "run_type_ratio": run_type_ratio
    }

# -------------------------------
# 3) 설명 문장 생성 로직
# -------------------------------
def generate_text_explanation(
    recent_stats, past_stats, 
    next_month_km=None, last_month_km=None
):
    messages = []

    # 1) 거리 추세
    if not np.isnan(recent_stats["mean_distance"]) and not np.isnan(past_stats["mean_distance"]):
        diff_dist = recent_stats["mean_distance"] - past_stats["mean_distance"]
        if diff_dist > 0.5:
            messages.append(f"최근 평균 러닝 거리(회당)가 이전보다 약 {diff_dist:.1f}km 늘었습니다.")
        elif diff_dist < -0.5:
            messages.append(f"최근 평균 러닝 거리(회당)가 이전보다 약 {abs(diff_dist):.1f}km 줄었습니다.")
        else:
            messages.append("최근 평균 러닝 거리는 이전과 비슷한 수준입니다.")

    # 2) 페이스 추세
    if not np.isnan(recent_stats["mean_pace"]) and not np.isnan(past_stats["mean_pace"]):
        diff_pace = recent_stats["mean_pace"] - past_stats["mean_pace"]
        if diff_pace < -0.1:
            messages.append(f"평균 페이스가 약 {-diff_pace:.2f}분/km 빨라졌어요. 체력이 좋아지는 중입니다.")
        elif diff_pace > 0.1:
            messages.append(f"평균 페이스가 약 {diff_pace:.2f}분/km 느려졌어요. 무리하지 않고 조절 중일 가능성이 있습니다.")
        else:
            messages.append("평균 페이스는 안정적으로 유지되고 있습니다.")

    # 3) 심박 효율
    if not np.isnan(recent_stats["mean_hr"]) and not np.isnan(past_stats["mean_hr"]):
        # 간단한 효율: 거리 / 심박
        recent_eff = recent_stats["mean_distance"] / recent_stats["mean_hr"]
        past_eff   = past_stats["mean_distance"] / past_stats["mean_hr"]
        eff_diff   = recent_eff - past_eff
        if eff_diff > 0.001:
            messages.append("같은 심박수로 더 먼 거리를 뛰고 있어, 심폐 지구력이 향상된 모습입니다.")
        elif eff_diff < -0.001:
            messages.append("최근에는 예전보다 심폐 효율이 약간 떨어진 상태입니다. 휴식과 컨디션 조절이 필요할 수 있어요.")
        else:
            messages.append("심박 대비 거리 효율은 큰 변화 없이 유지되고 있습니다.")

    # 4) run_type 비율 설명
    rt = recent_stats["run_type_ratio"]
    if rt:
        total = sum(rt.values())
        lsd_ratio = rt.get("LSD", 0)
        interval_ratio = rt.get("Interval", 0)
        long_ratio = rt.get("Long Run", 0)

        # 아주 단순한 멘트 예시
        if lsd_ratio > 0.5:
            messages.append("최근에는 LSD 위주의 훈련을 많이 하고 있습니다.")
        elif interval_ratio > 0.4:
            messages.append("최근에는 인터벌 비중이 높은 편입니다. 강도 높은 훈련이 많아요.")
        elif long_ratio > 0.4:
            messages.append("롱런 비중이 상대적으로 높은 편입니다. 장거리 지구력 향상에 도움이 됩니다.")

    # 5) LSTM 예측 반영
    if next_month_km is not None and last_month_km is not None:
        diff_month = next_month_km - last_month_km
        if diff_month > 5:
            messages.append(f"다음 달에는 이번 달보다 약 {diff_month:.1f}km 더 달릴 수 있을 것으로 예측됩니다.")
        elif diff_month < -5:
            messages.append(f"다음 달에는 이번 달보다 약 {abs(diff_month):.1f}km 적게 달릴 가능성이 있습니다.")
        else:
            messages.append("다음 달 러닝 거리 예측은 이번 달과 비슷한 수준입니다.")

    # summary 한 줄 뽑기 (첫 문장 기반)
    summary = messages[0] if messages else "최근 러닝 패턴이 안정적으로 유지되고 있습니다."

    return {
        "summary": summary,
        "detail": messages
    }

# -------------------------------
# 4) 전체 XAI 함수
# -------------------------------
def explain_running_pattern(history_df, next_month_km=None):
    """
    history_df: 한 유저의 running_record (하나의 user_id만 필터된 상태)
    next_month_km: LSTM이 예측한 다음 달 누적 거리 (없으면 None)
    """
    # 우선 start_time 기준 정렬
    history_df = history_df.sort_values("start_time")

    # 최근/이전 구간 나누기 (회수 기준)
    recent, past = split_recent_past(history_df, recent_n=5)

    recent_stats = compute_basic_stats(recent)
    past_stats   = compute_basic_stats(past)

    # 지난 달 실제 누적 거리 (간단 버전: 최근 30일 합)
    if not history_df.empty:
        last_30 = history_df[history_df["start_time"] >= history_df["start_time"].max() - pd.Timedelta(days=30)]
        last_month_km = last_30["distance_km"].sum()
    else:
        last_month_km = None

    result = generate_text_explanation(
        recent_stats, past_stats,
        next_month_km=next_month_km,
        last_month_km=last_month_km
    )
    return result


In [ ]:
# history_df는 이미 user_id로 필터한 running_record 데이터라고 가정
explanation = explain_running_pattern(history_df, next_month_km=52.3)

print(explanation["summary"])
for line in explanation["detail"]:
    print(" -", line)


In [ ]:
#LSTM 팀과의 연동 방식
def get_user_monthly_prediction_and_explanation(user_id):
    history_df = load_user_history_from_db(user_id)

    # 1) LSTM 팀 함수 호출 (우리는 이 안은 몰라도 됨)
    next_month_km = lstm_predict_next_month_km(history_df)

    # 2) XAI 설명 생성
    explanation = explain_running_pattern(history_df, next_month_km)

    # 3) 프론트에 넘길 JSON 형태
    return {
        "user_id": user_id,
        "next_month_km": float(next_month_km),
        "explanation": explanation
    }
